In [ ]:
import pandas as pd
import torch, numpy as np

# 1) Load & shuffle

In [ ]:
FullEvoKG = torch.load('1-Making_Indexing/Human_KG.pt')
N = FullEvoKG.size(0)
perm = torch.randperm(N)
shuffled = FullEvoKG[perm]

# 2) Split into test train and valid dataset

In [ ]:
# 2) Use [80,10,10] → total = 100
ratios = [80, 10, 10]
total  = sum(ratios)      # = 100
n_train = int(N * ratios[0] / total)  # 0.80 * N
n_valid = int(N * ratios[1] / total)  # 0.10 * N
n_test  = N - n_train - n_valid       # ensures sum = N

# 3) Slice the shuffled tensor
train = shuffled[:n_train]
valid = shuffled[n_train:n_train + n_valid]
test  = shuffled[n_train + n_valid:]

print(f"Train / Valid / Test = {len(train)}/{len(valid)}/{len(test)}")

# 4) Save out as tab-separated .txt
np.savetxt('Training_files/train_80.txt', train.cpu().numpy(), fmt='%d', delimiter='\t')
np.savetxt('Training_files/valid_10.txt', valid.cpu().numpy(), fmt='%d', delimiter='\t')
np.savetxt('Training_files/test_10.txt',  test.cpu().numpy(),  fmt='%d', delimiter='\t')

In [ ]:
Train = pd.read_csv('Training_files/train_final_80.txt',sep ='\t',header = None)
Train

In [ ]:
Test = pd.read_csv('Training_files/test_final_10.txt',sep ='\t',header = None)
Test

In [ ]:
valid = pd.read_csv('Training_files/valid_final_10.txt',sep ='\t',header = None)
valid

# 3) Making files as per DGL requirement

### Test Train Valid files

In [ ]:
Test = torch.load('Training_files/test_final_10.txt')
# 2. Write to a text file with tab‐separation:
with open("Training_files/Test_final.txt", "w", encoding="utf-8") as out:
    for head, rel, tail in Test.tolist():
        out.write(f"{head}\t{rel}\t{tail}\n")

Train = torch.load('AllKG_80_percent_training.pt')
# 2. Write to a text file with tab‐separation:
with open("Training_files/Train_final.txt", "w", encoding="utf-8") as out:
    for head, rel, tail in Train.tolist():
        out.write(f"{head}\t{rel}\t{tail}\n")

Valid = torch.load('AllKG_10_percent_valid.pt')
# 2. Write to a text file with tab‐separation:
with open("Training_files/valid_final.txt", "w", encoding="utf-8") as out:
    for head, rel, tail in Valid.tolist():
        out.write(f"{head}\t{rel}\t{tail}\n")

### Entities.dict files

In [ ]:
df = pd.read_pickle("Store_House/node_id_All_KG.pkl")  # or pd.read_pickle, etc.
df

In [ ]:
# 2. Select and reorder the two columns, drop any header, and write to file.
df[["MappedID", "Node"]] \
    .to_csv(
        "Training_files/entities.dict",
        sep="\t",         # space‐separated; use '\t' if you prefer tabs
        index=False,     # don’t write the pandas row‐index
        header=False,    # don’t write column names
        encoding="utf-8"
    )
df

In [ ]:
#!/usr/bin/env python3
import os

INPUT = "Training_files/entities.dict"
OUTPUT = "Training_files/entities_final.dict"

record = None
lines_written = 0

with open(INPUT, "r", encoding="utf-8") as infile, open(OUTPUT, "w", encoding="utf-8") as outfile:
    for line in infile:
        line = line.rstrip("\n")
        if line and line[0].isdigit():
            # If not the first record, write the accumulated record
            if record is not None:
                outfile.write(record + "\n")
                lines_written += 1
            record = line
        else:
            # Continuation line: append with a space
            if record is None:
                record = line  # safety case (shouldn’t normally happen)
            else:
                record += " " + line

    # Write the last record
    if record is not None:
        outfile.write(record + "\n")
        lines_written += 1

print(f"Repaired file written to {OUTPUT}")
print(f"Original lines:  {sum(1 for _ in open(INPUT, encoding='utf-8'))}")
print(f"Fixed   lines:  {lines_written}")


### relation.dict

In [ ]:
relations = {
    "phenotype_chemicalentity": 0,
    "mutation_disease": 1,
    "molecularfunction_chemicalentity": 2,
    "disease_anatomy": 3,
    "chemicalentity_disease": 4,
    "disease_disease": 5,
    "biologicalprocess_gene": 6,
    "protein_protein": 7,
    "gene_phenotype": 8,
    "protein_disease": 9,
    "anatomy_gene": 10,
    "chemicalentity_biologicalprocess": 11,
    "disease_gene": 12,
    "gene_cellularcomponent": 13,
    "chemicalentity_chemicalentity": 14,
    "cellularcomponent_gene": 15,
    "gene_disease": 16,
    "protein_cellularcomponent": 17,
    "protein_phenotype": 18,
    "mutation_protein": 19,
    "chemicalentity_gene": 20,
    "chemicalentity_tissue": 21,
    "chemicalentity_protein": 22,
    "biologicalprocess_biologicalprocess": 23,
    "phenotype_phenotype": 24,
    "phenotype_gene": 25,
    "chemicalentity_inhibits_biologicalprocess": 26,
    "gene_inhibits_biologicalprocess": 27,
    "protein_biologicalprocess": 28,
    "gene_promotes_biologicalprocess": 29,
    "gene_molecularfunction": 30,
    "gene_pathway": 31,
    "chemicalentity_pathway": 32,
    "gene_tissue": 33,
    "disease_phenotype": 34,
    "chemicalentity_mutation": 35,
    "gene_anatomy": 36,
    "phenotype_disease": 37,
    "pathway_gene": 38,
    "disease_chemicalentity": 39,
    "disease_mutation": 40,
    "gene_chemicalentity": 41,
    "protein_pathway": 42,
    "gene_protein": 43,
    "gene_noeffect_biologicalprocess": 44,
    "chemicalentity_promotes_biologicalprocess": 45,
    "gene_biologicalprocess": 46,
    "protein_molecularfunction": 47,
    "mutation_gene": 48,
    "gene_gene": 49,
    "molecularfunction_molecularfunction": 50,
    "gene_mutation": 51,
    "molecularfunction_biologicalprocess": 52,
    "protein_tissue": 53,
    "cellularcomponent_cellularcomponent": 54,
    "pathway_pathway": 55,
    "anatomy_anatomy": 56,
    "biologicalprocess_chemicalentity": 57,
    "plantextract_chemicalentity": 58,
    "plantextract_disease": 59,
    "pmid_cellularcomponent": 60,
    "pmid_chemicalentity": 61,
    "pmid_disease": 62,
    "pmid_protein": 63,
    "pmid_tissue": 64,
    "species_associatedwith_nodes": 65
}

output_path = "Training_files/relation_final.dict"

with open(output_path, "w", encoding="utf-8") as f:
    # Sort by ID so lines go 0,1,2,...
    for name, idx in sorted(relations.items(), key=lambda kv: kv[1]):
        f.write(f"{idx}\t{name}\n")

print(f"Wrote {len(relations)} lines to {output_path}")

### Deciding Max Step for DGL Training

In [ ]:
Epochs = 10
training_triples = 
batch_size = 2048
max_step = (Epochs * training_triples) / batch_size
max_step